# Day 9 - Tree-Based Models

This notebook contains all tasks from Day 9 covering decision trees, random forests, feature importance, and hyperparameter tuning.

## Task 8: Decision Tree & Random Forest - Wine Dataset

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt

wine = load_wine()

X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

dt_deep = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

dt_deep.fit(X_train, y_train)

print("Decision Tree (max_depth=None)")
print("Train Accuracy:", dt_deep.score(X_train, y_train))
print("Test Accuracy:", dt_deep.score(X_test, y_test))

dt_pruned = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt_pruned.fit(X_train, y_train)

print("\nDecision Tree (max_depth=3)")
print("Train Accuracy:", dt_pruned.score(X_train, y_train))
print("Test Accuracy:", dt_pruned.score(X_test, y_test))

estimators = [50, 100, 200]

models = {}

for n in estimators:

    rf = RandomForestClassifier(
        n_estimators=n,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    models[n] = rf

    print(f"\nRandom Forest ({n} trees)")
    print("Train Accuracy:", rf.score(X_train, y_train))
    print("Test Accuracy:", rf.score(X_test, y_test))
    print("OOB Score:", rf.oob_score_)

best_rf = models[200]

importance_df = pd.DataFrame({
    "Feature": wine.feature_names,
    "Importance": best_rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
).head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    importance_df["Feature"][::-1],
    importance_df["Importance"][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importances")

plt.tight_layout()
plt.show()

for n in estimators:

    rf = models[n]

    print(f"\nRandom Forest ({n} trees)")
    print("OOB Score:", rf.oob_score_)
    print("Test Accuracy:", rf.score(X_test, y_test))

## Task 9: Hyperparameter Tuning - Breast Cancer Dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

baseline_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

baseline_rf.fit(X_train, y_train)

baseline_test_acc = baseline_rf.score(X_test, y_test)

print("Baseline Test Accuracy:", baseline_test_acc)

param_grid = {
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV Score:")
print(grid.best_score_)

best_model = grid.best_estimator_

best_test_acc = best_model.score(X_test, y_test)

print("\nTuned Test Accuracy:")
print(best_test_acc)

print("\nImprovement:")
print(best_test_acc - baseline_test_acc)